In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


train = pd.read_csv("../1-Data/train_processed.csv")
x_train_before = train.drop(columns=["HeartDisease"])
y_train = train["HeartDisease"]

test = pd.read_csv("../1-Data/test_processed.csv")
x_test_before = test.drop(columns=["HeartDisease"])
y_test = test["HeartDisease"]


CHOSEN_C = 0.01
CHOSEN_THRESHOLD = 0.45

# ==========================================
# PHASE 1: MODEL EVALUATION (BEFORE FEATURE)
# ==========================================
model_before = LogisticRegression(C=CHOSEN_C, random_state=42, max_iter=1000)
model_before.fit(x_train_before, y_train)

y_proba_before = model_before.predict_proba(x_test_before)[:, 1]
y_pred_before = (y_proba_before >= CHOSEN_THRESHOLD).astype(int)
acc_before = accuracy_score(y_test, y_pred_before)

# ==========================================
# PHASE 2: IMPLEMENT THE LLM FEATURE (AFTER)
# ==========================================
x_train_after = x_train_before.copy()
x_test_after = x_test_before.copy()


x_train_after["Age_x_MaxHR"] = x_train_after["Age"] * x_train_after["MaxHR"]
x_test_after["Age_x_MaxHR"] = x_test_after["Age"] * x_test_after["MaxHR"]

model_after = LogisticRegression(C=CHOSEN_C, random_state=42, max_iter=1000)
model_after.fit(x_train_after, y_train)

y_proba_after = model_after.predict_proba(x_test_after)[:, 1]
y_pred_after = (y_proba_after >= CHOSEN_THRESHOLD).astype(int)
acc_after = accuracy_score(y_test, y_pred_after)

# ==========================================
# PHASE 3: METRIC COMPARISON OUTPUT
# ==========================================
print("\n=========================================================")
print("          BEFORE VS. AFTER CONTROLLED BENCHMARK          ")
print("=========================================================")
print(f"Accuracy BEFORE LLM Feature: {acc_before * 100:.2f}%")
print(f"Accuracy AFTER LLM Feature:  {acc_after * 100:.2f}%")
print(f"Absolute Performance Impact: {((acc_after - acc_before) * 100):+.2f}%")
print("=========================================================\n")

print("--- CLASSIFICATION REPORT BEFORE FEATURE ---")
print(classification_report(y_test, y_pred_before, target_names=["Normal", "Heart Disease"]))

print("--- CLASSIFICATION REPORT AFTER FEATURE ---")
print(classification_report(y_test, y_pred_after, target_names=["Normal", "Heart Disease"]))


          BEFORE VS. AFTER CONTROLLED BENCHMARK          
Accuracy BEFORE LLM Feature: 88.59%
Accuracy AFTER LLM Feature:  87.50%
Absolute Performance Impact: -1.09%

--- CLASSIFICATION REPORT BEFORE FEATURE ---
               precision    recall  f1-score   support

       Normal       0.91      0.83      0.87        82
Heart Disease       0.87      0.93      0.90       102

     accuracy                           0.89       184
    macro avg       0.89      0.88      0.88       184
 weighted avg       0.89      0.89      0.89       184

--- CLASSIFICATION REPORT AFTER FEATURE ---
               precision    recall  f1-score   support

       Normal       0.89      0.82      0.85        82
Heart Disease       0.86      0.92      0.89       102

     accuracy                           0.88       184
    macro avg       0.88      0.87      0.87       184
 weighted avg       0.88      0.88      0.87       184

